# Healthcare Claims — ETL Pipeline

Ingesting, validating, and cleaning a synthetic Medicaid-style claims dataset.

The main things I wanted this to handle robustly:
- Missing or null values in required fields
- Invalid claim type / status codes
- Amount fields that aren't actually numeric
- Denied claims that still have a paid amount (data anomaly)

Output: clean CSV + QA error log + analytics summary JSON.

In [1]:
import pandas as pd
import numpy as np
import json
import os
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## 1. Load

In [2]:
df = pd.read_csv('../data/sample_claims.csv', dtype=str)
print(f'{len(df)} rows, {len(df.columns)} columns')
df.head()

20 rows, 14 columns


,claim_id,member_id,state_code,plan_id,service_date,claim_type,diagnosis_code,procedure_code,billed_amount,paid_amount,claim_status,provider_id,provider_type,managed_care_flag
0,CLM0001,MBR1001,TX,MCO_TX_01,2024-01-05,IP,J18.9,99223,12500.00,9800.00,PAID,PRV5001,HOSPITAL,Y
1,CLM0002,MBR1002,TX,MCO_TX_01,2024-01-07,OP,Z00.00,99213,250.00,180.00,PAID,PRV5002,PHYSICIAN,Y
2,CLM0003,MBR1003,FL,MCO_FL_02,2024-01-08,OP,E11.9,99214,350.00,270.00,PAID,PRV5003,PHYSICIAN,Y
3,CLM0004,MBR1004,FL,MCO_FL_02,2024-01-10,IP,I21.9,99223,18000.00,14200.00,PAID,PRV5004,HOSPITAL,Y
4,CLM0005,MBR1005,CA,MCO_CA_03,2024-01-12,OP,F32.1,90837,200.00,150.00,PAID,PRV5005,BEHAVIORAL,Y


In [3]:
df.dtypes

claim_id             object
member_id            object
state_code           object
plan_id              object
service_date         object
claim_type           object
diagnosis_code       object
procedure_code       object
billed_amount        object
paid_amount          object
claim_status         object
provider_id          object
provider_type        object
managed_care_flag    object
dtype: object

## 2. Validate

Checking for nulls, invalid codes, and non-numeric amounts. Rows that fail go to a separate error log.

In [4]:
VALID_CLAIM_TYPES = {'IP', 'OP', 'LTC', 'RX'}
VALID_STATUSES    = {'PAID', 'DENIED', 'PENDING', 'VOID'}
REQUIRED_COLS     = ['claim_id','member_id','state_code','service_date',
                     'claim_type','billed_amount','paid_amount','claim_status']

errors       = []
invalid_mask = pd.Series(False, index=df.index)

# Null checks
for col in REQUIRED_COLS:
    null_rows = df[col].isna()
    for idx in df[null_rows].index:
        errors.append({'row': idx, 'claim_id': df.loc[idx,'claim_id'], 'error': f'Null in {col}'})
    invalid_mask |= null_rows

# Claim type
bad_type = ~df['claim_type'].isin(VALID_CLAIM_TYPES)
for idx in df[bad_type].index:
    errors.append({'row': idx, 'claim_id': df.loc[idx,'claim_id'], 'error': f"Unknown claim_type: {df.loc[idx,'claim_type']}"})
invalid_mask |= bad_type

# Status
bad_status = ~df['claim_status'].isin(VALID_STATUSES)
for idx in df[bad_status].index:
    errors.append({'row': idx, 'claim_id': df.loc[idx,'claim_id'], 'error': f"Unknown status: {df.loc[idx,'claim_status']}"})
invalid_mask |= bad_status

# Numeric amounts
for col in ['billed_amount','paid_amount']:
    bad_num = pd.to_numeric(df[col], errors='coerce').isna()
    for idx in df[bad_num].index:
        errors.append({'row': idx, 'claim_id': df.loc[idx,'claim_id'], 'error': f'Non-numeric {col}'})
    invalid_mask |= bad_num

clean_df = df[~invalid_mask].copy()
error_df = pd.DataFrame(errors)

print(f'Valid: {len(clean_df)} | Invalid: {len(df[invalid_mask])}')
if not error_df.empty:
    display(error_df)

Valid: 20 | Invalid: 0


## 3. Transform

In [5]:
clean_df['service_date']    = pd.to_datetime(clean_df['service_date'])
clean_df['billed_amount']   = pd.to_numeric(clean_df['billed_amount'])
clean_df['paid_amount']     = pd.to_numeric(clean_df['paid_amount'])

clean_df['service_year']    = clean_df['service_date'].dt.year
clean_df['service_month']   = clean_df['service_date'].dt.month
clean_df['service_quarter'] = clean_df['service_date'].dt.quarter

clean_df['payment_rate'] = np.where(
    clean_df['billed_amount'] > 0,
    (clean_df['paid_amount'] / clean_df['billed_amount']).round(4),
    0.0
)

clean_df['is_inpatient']    = clean_df['claim_type']    == 'IP'
clean_df['is_denied']       = clean_df['claim_status']  == 'DENIED'
clean_df['is_behavioral']   = clean_df['provider_type'] == 'BEHAVIORAL'

# Flag: denied but still has a paid amount
clean_df['anomaly_paid_denied'] = clean_df['is_denied'] & (clean_df['paid_amount'] > 0)

n = clean_df['anomaly_paid_denied'].sum()
if n:
    print(f'⚠  {n} denied claims with paid_amount > 0')

clean_df.head()

,claim_id,member_id,state_code,plan_id,service_date,claim_type,diagnosis_code,procedure_code,billed_amount,paid_amount,claim_status,provider_id,provider_type,managed_care_flag,service_year,service_month,service_quarter,payment_rate,is_inpatient,is_denied,is_behavioral,anomaly_paid_denied
0,CLM0001,MBR1001,TX,MCO_TX_01,2024-01-05,IP,J18.9,99223,"12,500.00","9,800.00",PAID,PRV5001,HOSPITAL,Y,2024,1,1,0.78,True,False,False,False
1,CLM0002,MBR1002,TX,MCO_TX_01,2024-01-07,OP,Z00.00,99213,250.00,180.00,PAID,PRV5002,PHYSICIAN,Y,2024,1,1,0.72,False,False,False,False
2,CLM0003,MBR1003,FL,MCO_FL_02,2024-01-08,OP,E11.9,99214,350.00,270.00,PAID,PRV5003,PHYSICIAN,Y,2024,1,1,0.77,False,False,False,False
3,CLM0004,MBR1004,FL,MCO_FL_02,2024-01-10,IP,I21.9,99223,"18,000.00","14,200.00",PAID,PRV5004,HOSPITAL,Y,2024,1,1,0.79,True,False,False,False
4,CLM0005,MBR1005,CA,MCO_CA_03,2024-01-12,OP,F32.1,90837,200.00,150.00,PAID,PRV5005,BEHAVIORAL,Y,2024,1,1,0.75,False,False,True,False


## 4. Analytics Summary

In [6]:
paid = clean_df[clean_df['claim_status'] == 'PAID']

print(f"Total claims:      {len(clean_df)}")
print(f"Paid:              {len(paid)}")
print(f"Denied:            {clean_df['is_denied'].sum()}")
print(f"Denial rate:       {clean_df['is_denied'].mean()*100:.1f}%")
print(f"Total billed:      ${clean_df['billed_amount'].sum():,.2f}")
print(f"Total paid:        ${clean_df['paid_amount'].sum():,.2f}")
print(f"Avg payment rate:  {paid['payment_rate'].mean()*100:.1f}%")

Total claims:      20
Paid:              18
Denied:            2
Denial rate:       10.0%
Total billed:      $78,150.00
Total paid:        $57,720.00
Avg payment rate:  75.7%


In [7]:
# Spend by state
clean_df.groupby('state_code')['paid_amount'].sum().sort_values(ascending=False).rename('total_paid')

state_code
FL   26,730.00
TX   19,140.00
CA   11,850.00
Name: total_paid, dtype: float64

In [8]:
# Spend by claim type
clean_df.groupby('claim_type')['paid_amount'].agg(['sum','mean','count']).round(2)

,sum,mean,count
claim_type,,,
IP,"51,800.00","10,360.00",5
LTC,"3,800.00","1,900.00",2
OP,"2,120.00",163.08,13


In [9]:
# Denial rate by provider type
(
    clean_df.groupby('provider_type')
    .agg(total=('claim_id','count'), denied=('is_denied','sum'))
    .assign(denial_rate=lambda x: (x['denied']/x['total']*100).round(1))
    .sort_values('denial_rate', ascending=False)
)

,total,denied,denial_rate
provider_type,,,
NURSING,2,1,50.00
PHYSICIAN,10,1,10.00
BEHAVIORAL,3,0,0.00
HOSPITAL,5,0,0.00


In [10]:
# Top diagnosis codes by spend
(
    paid.groupby('diagnosis_code')['paid_amount']
    .agg(['sum','count'])
    .rename(columns={'sum':'total_paid','count':'claims'})
    .sort_values('total_paid', ascending=False)
    .head(10)
)

,total_paid,claims
diagnosis_code,,
J18.9,"18,400.00",2
I21.9,"14,200.00",1
I50.9,"11,800.00",1
K35.80,"7,400.00",1
F03.90,"3,800.00",1
E11.9,920.00,4
Z00.00,590.00,4
F32.1,450.00,3
J06.9,160.00,1


## 5. Export

In [11]:
os.makedirs('../data/output', exist_ok=True)
clean_df.to_csv('../data/output/claims_clean.csv', index=False)
if not error_df.empty:
    error_df.to_csv('../data/output/validation_errors.csv', index=False)
print('Exported.')

Exported.
